Scanning Optimization
----------------------
- Being a Data Engineer we need to prepare data destinations 
- There are multiple files, there will be a data analyst that'll need to query the data.
- Everytime they query the data they will need to scan all the files, given there is no optimization in the data.
- To optimize the scanning can use PartionBy

In [0]:
from pyspark.sql.functions import *

In [0]:
# from pyspark.sql.functions import col

# df = (
#     spark.read.format("csv")
#     .option("header", "true")
#     .load("/Volumes/sparkoptimization/tables/epl_volume/*.csv")
#     .withColumn("file_name", col("_metadata.file_path"))
# )

# df.write.mode("append").saveAsTable("sparkoptimization.tables.epl_records")

#Turn Off AQE
----------------
Normally Spark:

optimises joins at runtime
changes shuffle partitions automatically
improves performance dynamically

When you disable it , you are telling spark
“Do NOT change my query plan at runtime — stick to the original plan.”

| Setting     | Effect                                      |
| ----------- | ------------------------------------------- |
| AQE = true  | Spark optimises automatically (recommended) |
| AQE = false | You control execution plan manually         |


In [0]:
# spark.conf.set("spark.sql.adaptive.enabled", "false")
# Note: spark.sql.adaptive.enabled is not configurable on serverless compute
# Serverless automatically manages Adaptive Query Execution (AQE)

In [0]:
# # Checking AQE Status
# spark.conf.get("spark.sql.adaptive.enabled")
# Would return true

# Read Data
------------

In [0]:

##
df = spark.read.format("csv").option('inferSchema', True).option("header", True).load("/Volumes/sparkoptimization/tables/epl_volume/*.csv")

display(df)

In [0]:
# df.rdd.getNumPartitions() ===> Would give the number of partitions spark has set by default
### Spark is using Adaptive Query Execution (AQE) to automatically decide the number of shuffle partitions at runtime.
spark.conf.get("spark.sql.shuffle.partitions")

# Change DEFAULT Partition Size ti 128kb
------------------------------------------

In [0]:
# changinfg the default partition size to 128kb

spark.conf.set("spark.sql.files.maxPartitionBytes", 131072)

display(spark.conf.get("spark.sql.files.maxPartitionBytes"))

In [0]:
# display(df.rdd.getNumPartitions())

# Get partition information
-------------------------------

In [0]:
### There are currently 10 partitions
display(df.withColumn('partition_id', spark_partition_id()).groupBy(col('partition_id')).count())

In [0]:
### Load all of the 12 files present in the volume
spark.read.format("csv").option('inferSchema', True).option("header", True).load("/Volumes/sparkoptimization/tables/epl_volume/*.csv").display()

# SCANNING OPTIMIZATION
-----------------------------

In [0]:
### Add a partitionBy based on time
df.write.mode('append').partitionBy('time').saveAsTable('sparkoptimization.tables.partitioned_epl_records')